In [53]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/himanshubendale000/alllset/merged_dataset_ds003458.pkl
/kaggle/input/datasets/himanshubendale000/alllset/all_subjects_epochs_ds003458_v2 (1).pkl
/kaggle/input/datasets/himanshubendale000/alllset/oos_3arm_comparison (1).csv
/kaggle/input/datasets/himanshubendale000/alllset/merged_dataset_ds003458 (1).pkl
/kaggle/input/datasets/himanshubendale000/alllset/rpe_traces_ds003458.pkl
/kaggle/input/datasets/himanshubendale000/alllset/all_subjects_epochs_ds003458 (1).pkl
/kaggle/input/datasets/himanshubendale000/alllset/cnn_predictions_seed0.pkl
/kaggle/input/datasets/himanshubendale000/alllset/all_subjects_epochs_ds003458.pkl
/kaggle/input/datasets/himanshubendale000/alllset/aic_3arm_comparison (1).csv
/kaggle/input/datasets/himanshubendale000/alllset/rpe_traces_ds003458 (1).pkl
/kaggle/input/datasets/himanshubendale000/alllset/all_subjects_epochs_ds003458_v2.pkl
/kaggle/input/datasets/himanshubendale000/alllset/ds003458_B9_pipeline.ipynb
/kaggle/input/datasets/himanshube

In [35]:
# ===== Cell 0 — Kaggle boilerplate + imports =====
import os, gc, pickle
from pathlib import Path

import numpy as np
import pandas as pd

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/himanshubendale000/alllset/merged_dataset_ds003458.pkl
/kaggle/input/datasets/himanshubendale000/alllset/all_subjects_epochs_ds003458_v2 (1).pkl
/kaggle/input/datasets/himanshubendale000/alllset/oos_3arm_comparison (1).csv
/kaggle/input/datasets/himanshubendale000/alllset/merged_dataset_ds003458 (1).pkl
/kaggle/input/datasets/himanshubendale000/alllset/rpe_traces_ds003458.pkl
/kaggle/input/datasets/himanshubendale000/alllset/all_subjects_epochs_ds003458 (1).pkl
/kaggle/input/datasets/himanshubendale000/alllset/cnn_predictions_seed0.pkl
/kaggle/input/datasets/himanshubendale000/alllset/all_subjects_epochs_ds003458.pkl
/kaggle/input/datasets/himanshubendale000/alllset/aic_3arm_comparison (1).csv
/kaggle/input/datasets/himanshubendale000/alllset/rpe_traces_ds003458 (1).pkl
/kaggle/input/datasets/himanshubendale000/alllset/all_subjects_epochs_ds003458_v2.pkl
/kaggle/input/datasets/himanshubendale000/alllset/ds003458_B9_pipeline.ipynb
/kaggle/input/datasets/himanshube

In [36]:
# ===== Cell 1 — Checkpoint infra: find() / cached() / cached_incremental() =====
# find(): locate a file anywhere under /kaggle/input or /kaggle/working, however nested.
# cached(): whole-object cache — build once, reuse forever (good for cheap/medium steps).
# cached_incremental(): per-key cache — saves after EVERY key, so a heavy loop over
#   23 subjects (or subject×region×model combos) can be killed/restarted and will
#   only recompute the keys that are still missing.

INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)

INDEX = {}
def _scan(root):
    if not root.exists():
        return
    for dp, _, fs in os.walk(root, followlinks=True):
        for f in fs:
            INDEX.setdefault(f, []).append(Path(dp) / f)

_scan(INPUT_ROOT)
_scan(WORK)
print(f"indexed {sum(len(v) for v in INDEX.values())} files "
      f"({len(INDEX)} unique names) under /kaggle/input + /kaggle/working")

def find(name):
    hits = INDEX.get(name, [])
    if not hits:
        return None
    return sorted(hits, key=lambda p: 0 if str(p).startswith(str(WORK)) else 1)[0]

def cached(name, builder):
    """Whole-object cache. Loads name if it exists anywhere; else builds once and saves."""
    p = find(name)
    if p is not None:
        try:
            if p.suffix == ".csv":
                obj = pd.read_csv(p)
            else:
                with open(p, "rb") as f:
                    obj = pickle.load(f)
            print(f"✔ {name:<40} ← loaded: {p}")
            return obj
        except Exception as e:
            print(f"⚠ {name} failed to load ({e}) — rebuilding")

    print(f"… building {name} (not found in cache)")
    obj = builder()
    out = WORK / name
    if isinstance(obj, pd.DataFrame) and name.endswith(".csv"):
        obj.to_csv(out, index=False)
    else:
        with open(out, "wb") as f:
            pickle.dump(obj, f)
    INDEX.setdefault(name, []).append(out)
    print(f"💾 saved {name} → {out}")
    return obj

def cached_incremental(name, keys, build_one, save_every=1):
    """
    Per-key resumable cache. `keys` is the full list of things that must end up
    in the result dict (e.g. subject ids, or 'region__model' strings).
    `build_one(key)` computes ONE entry. Progress is written to disk after every
    `save_every` new keys, so an interrupted run resumes without recomputing
    anything already done.
    """
    p = find(name)
    result = {}
    if p is not None:
        try:
            with open(p, "rb") as f:
                result = pickle.load(f)
            print(f"✔ {name:<40} ← resuming from: {p} ({len(result)}/{len(keys)} keys already done)")
        except Exception as e:
            print(f"⚠ {name} failed to load ({e}) — starting fresh")
            result = {}

    out_path = WORK / name
    n_new = 0
    for i, k in enumerate(keys):
        if k in result:
            continue
        result[k] = build_one(k)
        n_new += 1
        if n_new % save_every == 0:
            with open(out_path, "wb") as f:
                pickle.dump(result, f)
    # final save (covers any remainder + the "nothing new" case still verifies file exists)
    with open(out_path, "wb") as f:
        pickle.dump(result, f)
    INDEX.setdefault(name, []).append(out_path)
    print(f"💾 {name}: {len(result)}/{len(keys)} keys complete → {out_path}")
    return result

indexed 59 files (53 unique names) under /kaggle/input + /kaggle/working


In [4]:
# ===== B9 — Get the real file listing from the S3 bucket (no guessing, verify it) =====
!pip install boto3 --quiet

import boto3
from botocore import UNSIGNED
from botocore.config import Config

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED), region_name="us-east-1")

BUCKET = "openneuro.org"
PREFIX = "ds003458/"

print(f"=== structure inside s3://{BUCKET}/{PREFIX} (first ~50 items) ===")
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, Delimiter="/")

print("\n--- Top-level folders ---")
for cp in resp.get("CommonPrefixes", []):
    print(" ", cp["Prefix"])

print("\n--- Top-level files ---")
for obj in resp.get("Contents", []):
    print(" ", obj["Key"], f"({obj['Size']} bytes)")

# ---- Look inside sub-001 (where events.tsv, channels.tsv are and under what names) ----
print(f"\n=== inside s3://{BUCKET}/{PREFIX}sub-001/ (recursively) ===")
resp2 = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{PREFIX}sub-001/")
for obj in resp2.get("Contents", []):
    print(" ", obj["Key"], f"({obj['Size']} bytes)")

=== s3://openneuro.org/ds003458/ मधलं structure (पहिले ~50 items) ===

--- टॉप-लेव्हल फोल्डर्स ---
  ds003458/.datalad/
  ds003458/code/
  ds003458/stimuli/
  ds003458/sub-001/
  ds003458/sub-002/
  ds003458/sub-003/
  ds003458/sub-004/
  ds003458/sub-005/
  ds003458/sub-006/
  ds003458/sub-007/
  ds003458/sub-008/
  ds003458/sub-009/
  ds003458/sub-010/
  ds003458/sub-011/
  ds003458/sub-012/
  ds003458/sub-013/
  ds003458/sub-014/
  ds003458/sub-015/
  ds003458/sub-016/
  ds003458/sub-017/
  ds003458/sub-018/
  ds003458/sub-019/
  ds003458/sub-020/
  ds003458/sub-021/
  ds003458/sub-022/
  ds003458/sub-023/

--- टॉप-लेव्हल फाईल्स ---
  ds003458/.gitattributes (284 bytes)
  ds003458/CHANGES (46 bytes)
  ds003458/README (894 bytes)
  ds003458/annex-uuid (36 bytes)
  ds003458/dataset_description.json (291 bytes)
  ds003458/participants.json (280 bytes)
  ds003458/participants.tsv (322 bytes)

=== s3://openneuro.org/ds003458/sub-001/ मधलं (recursively) ===
  ds003458/sub-001/eeg/sub-001_

In [37]:
# ===== Cell 2 — S3 client (OpenNeuro public bucket, unsigned) =====
!pip install boto3 --quiet

import boto3
from botocore import UNSIGNED
from botocore.config import Config

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED), region_name="us-east-1")
BUCKET = "openneuro.org"
DS = "ds003458"
PREFIX = f"{DS}/"

resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, Delimiter="/")
n_subjects = sum(1 for cp in resp.get("CommonPrefixes", []) if "/sub-" in cp["Prefix"])
print(f"✔ reached s3://{BUCKET}/{PREFIX} — {n_subjects} subject folders visible")

✔ reached s3://openneuro.org/ds003458/ — 23 subject folders visible


In [38]:
# ===== Cell 3 — Reference/metadata files (participants, README, task info) =====
OUT = Path("/kaggle/working/ds003458_probe")
OUT.mkdir(parents=True, exist_ok=True)

REF_FILES = {
    "participants.tsv":         f"{DS}/participants.tsv",
    "README":                   f"{DS}/README",
    "dataset_description.json": f"{DS}/dataset_description.json",
    "README_Code.txt":          f"{DS}/code/README _ Code.txt",
}

def _build_ref_files():
    got = {}
    for local_name, key in REF_FILES.items():
        dest = OUT / local_name
        if not dest.exists():
            s3.download_file(BUCKET, key, str(dest))
        got[local_name] = str(dest)
    return got

ref_paths = cached("ds003458_ref_files.pkl", _build_ref_files)

participants = pd.read_csv(ref_paths["participants.tsv"], sep="\t")
print(participants.to_string(index=False))
print("\n--- README ---")
print(Path(ref_paths["README"]).read_text(errors="ignore"))

… building ds003458_ref_files.pkl (not found in cache)
💾 saved ds003458_ref_files.pkl → /kaggle/working/ds003458_ref_files.pkl
participant_id  sex  age
       sub-001    1   20
       sub-002    2   18
       sub-003    1   20
       sub-004    2   18
       sub-005    2   21
       sub-006    2   19
       sub-007    2   19
       sub-008    2   20
       sub-009    1   19
       sub-010    2   21
       sub-011    2   19
       sub-012    2   19
       sub-013    2   18
       sub-014    1   23
       sub-015    1   21
       sub-016    1   22
       sub-017    2   19
       sub-018    1   22
       sub-019    2   24
       sub-020    2   19
       sub-021    2   19
       sub-022    2   20
       sub-023    2   19

--- README ---
Healthy control college students.  23 subjects completed the 3-armed bandit task with oscillating probabilities.    For example, the 'blue' stim would slowly move from 20% reinforcing to 90% then back to 20 over many trials.  The other 'red' and 'green' sti

In [39]:
# ===== Cell 4 — Trigger-code resolver + EEG region definitions =====
# Bandit stim codes (from README_Code.txt): concatenation of [X=1,Y=2,Z=3] in
# [up,left,right] positions, then +100 to make it a clean 3-digit code.
#   1=up button, 2=left button, 3=right button, 4=no-response, 5=win(+1), 6=loss(~)

def find_event_code(event_id_dict, number):
    """Resolve a trigger NUMBER to whatever key MNE actually assigned it
    (padding like 'S  1' vs 'S100' varies), by comparing digits only."""
    target = str(number)
    for k, v in event_id_dict.items():
        digits = "".join(ch for ch in k if ch.isdigit())
        if digits == target:
            return v
    raise KeyError(f"trigger code {number} not found in event_id")

STIM_TRIGGER_NUMS = [23, 32, 113, 131, 212, 221]

FRONTAL = ['Fp1','Fz','F3','F7','F4','F8','Fp2','AF7','AF3','AFz','F1','F5','F6','F2','AF4','AF8']
CENTRAL = ['FC5','FC1','C3','CP5','CP1','CP6','CP2','Cz','C4','FC6','FC2','FC3','FCz','C1','C5','CP3','CP4','C6','C2','FC4']
PARIETO_OCCIPITAL = ['Pz','P3','P7','O1','Oz','O2','P4','P8','P1','P5','PO7','PO3','POz','PO4','PO8','P6','P2']
EXCLUDED = ['EOG','FT9','T7','TP9','TP10','T8','FT10','FT7','TP7','TP8','FT8']
REGIONS = ["frontal", "central", "parieto_occipital"]

print(f"Frontal={len(FRONTAL)}, Central={len(CENTRAL)}, Parieto-occipital={len(PARIETO_OCCIPITAL)}, "
      f"Excluded={len(EXCLUDED)}  (total defined={len(FRONTAL)+len(CENTRAL)+len(PARIETO_OCCIPITAL)+len(EXCLUDED)}/64)")

Frontal=16, Central=20, Parieto-occipital=17, Excluded=11  (total defined=64/64)


In [40]:
# ===== Cell 5 — Behavioral parser: events.tsv → per-trial choice/reward/RT =====
# RT = Response-onset - Stimulus-onset (events.tsv's own response_time column is empty).
# Verified 100% match against the raw-annotation parse on sub-001 (chosen_stim + reward).

def parse_bandit_trials(ev):
    rows, trial_num = [], 0
    pending_stim_map = pending_choice = stim_onset = resp_onset = None

    for _, r in ev.iterrows():
        tt, val = str(r["trial_type"]), str(r["value"]).strip()

        if tt.startswith("Stimulus"):
            code_num = int(val.replace("S", "").strip()) + 100
            d = str(code_num)
            up, left, right = int(d[0]), int(d[1]), int(d[2])
            pending_stim_map = {"Up": up, "Left": left, "Right": right}
            pending_choice, stim_onset, resp_onset = None, r["onset"], None

        elif tt.startswith("Response"):
            pos = tt.split(":")[1].strip()
            resp_onset = r["onset"]
            pending_choice = None if (pos == "No Response" or pending_stim_map is None) \
                              else pending_stim_map.get(pos)

        elif tt.startswith("Feedback"):
            reward = 1 if "Win" in tt else (0 if "Loss" in tt else np.nan)
            if pending_choice is not None and not pd.isna(reward):
                trial_num += 1
                rt = (resp_onset - stim_onset) if (resp_onset is not None and stim_onset is not None) else np.nan
                rows.append({"trial_num": trial_num, "chosen_stim": pending_choice,
                             "reward": int(reward), "rt": rt, "feedback_onset": r["onset"]})
            pending_stim_map = pending_choice = stim_onset = resp_onset = None

    return pd.DataFrame(rows)


def _build_one_subject_trials(subj):
    dest = OUT / f"{subj}_events.tsv"
    key = f"{DS}/{subj}/eeg/{subj}_task-ThreeArmedBandit_events.tsv"
    if not dest.exists():
        s3.download_file(BUCKET, key, str(dest))
    ev = pd.read_csv(dest, sep="\t")
    t = parse_bandit_trials(ev)
    print(f"  ✔ {subj}: {len(t)} trials, win-rate={t['reward'].mean():.3f}")
    return t

all_subjects = sorted(participants["participant_id"].tolist())
all_trials = cached_incremental("all_subjects_trials_ds003458.pkl", all_subjects, _build_one_subject_trials)

summ = pd.DataFrame([{"subject": s, "n_trials": len(t), "win_rate": t["reward"].mean()}
                     for s, t in all_trials.items()])
print(summ.round(3).to_string(index=False))

✔ all_subjects_trials_ds003458.pkl         ← resuming from: /kaggle/working/ds003458_probe/all_subjects_trials_ds003458.pkl (23/23 keys already done)
💾 all_subjects_trials_ds003458.pkl: 23/23 keys complete → /kaggle/working/all_subjects_trials_ds003458.pkl
subject  n_trials  win_rate
sub-001       479     0.752
sub-002       478     0.812
sub-003       480     0.821
sub-004       478     0.839
sub-005       480     0.790
sub-006       480     0.850
sub-007       479     0.783
sub-008       478     0.845
sub-009       479     0.770
sub-010       477     0.866
sub-011       480     0.850
sub-012       480     0.812
sub-013       479     0.835
sub-014       479     0.835
sub-015       480     0.798
sub-016       480     0.852
sub-017       480     0.794
sub-018       478     0.816
sub-019       479     0.812
sub-020       477     0.784
sub-021       476     0.796
sub-022       480     0.825
sub-023       480     0.773


In [41]:
# ===== Cell 6 — RL model definitions: M1 (RW), M2 (dual-LR), M3 (Pearce-Hall) =====
from scipy.optimize import minimize

def run_model_3arm(model, p, choices, rewards):
    """model in {'m1','m2','m3'}; p[-1] is always beta (softmax temperature)."""
    Q = np.array([0.5, 0.5, 0.5])
    A = np.array([0.5, 0.5, 0.5])  # only used by m3
    nll, n = 0.0, len(choices)
    rpes = np.zeros(n)

    for t in range(n):
        ch = choices[t]
        exp_q = np.exp(np.clip(p[-1] * (Q - Q.max()), -500, 500))
        probs = exp_q / exp_q.sum()
        nll -= np.log(max(probs[ch], 1e-10))

        rpe = rewards[t] - Q[ch]
        rpes[t] = rpe

        if model == "m1":
            Q[ch] += p[0] * rpe
        elif model == "m2":
            alpha = p[0] if rewards[t] == 1 else p[1]
            Q[ch] += alpha * rpe
        else:  # m3 — Pearce-Hall: learning rate itself adapts to |RPE|
            a_t = np.clip(p[0] + p[1] * A[ch], 0.001, 0.999)
            Q[ch] += a_t * rpe
            A[ch] = p[2] * abs(rpe) + (1 - p[2]) * A[ch]

    return nll, rpes

PARAMS = {"m1": ["alpha", "beta"],
          "m2": ["alpha_gain", "alpha_loss", "beta"],
          "m3": ["alpha0", "kappa", "eta", "beta"]}
BOUNDS = {"m1": [(0.001, 0.999), (0.01, 20)],
          "m2": [(0.001, 0.999), (0.001, 0.999), (0.01, 20)],
          "m3": [(0.0, 1.0), (0.0, 1.0), (0.001, 0.999), (0.01, 20)]}

def _x0(model, rng):
    if model == "m1": return [rng.uniform(0.05, 0.95), rng.uniform(0.1, 10.0)]
    if model == "m2": return [rng.uniform(0.05, 0.95), rng.uniform(0.05, 0.95), rng.uniform(0.1, 10.0)]
    return [rng.uniform(0.05, 0.5), rng.uniform(0.05, 0.95), rng.uniform(0.05, 0.95), rng.uniform(0.1, 10.0)]

def fit_model_3arm(model, trials_df, n_starts=10, seed=42):
    choices = (trials_df["chosen_stim"].values - 1).astype(int)
    rewards = trials_df["reward"].values.astype(float)
    rng = np.random.default_rng(seed)
    best_fun, best_x = np.inf, None
    for _ in range(n_starts):
        x0 = _x0(model, rng)
        res = minimize(lambda p: run_model_3arm(model, p, choices, rewards)[0],
                        x0=x0, method="Nelder-Mead", bounds=BOUNDS[model])
        if res.success and res.fun < best_fun:
            best_fun, best_x = res.fun, res.x
    return best_x, best_fun

In [42]:
# ===== Cell 7 — Fit M1 on all subjects (behavioral only, cheap) =====
def _build_m1_fits():
    rows = []
    for s, t in all_trials.items():
        x, nll = fit_model_3arm("m1", t)
        chance_nll = len(t) * np.log(3)
        rows.append({"subject": s, "n_trials": len(t), "alpha": x[0], "beta": x[1],
                     "nll": nll, "chance_nll": chance_nll, "above_chance": nll < chance_nll - 5})
        print(f"  {s}: alpha={x[0]:.3f}, beta={x[1]:.3f}, nll={nll:.2f}")
    return pd.DataFrame(rows)

fits_df = cached("m1_3arm_fits.csv", _build_m1_fits)
print(f"\nAbove chance: {fits_df['above_chance'].sum()}/{len(fits_df)}")

✔ m1_3arm_fits.csv                         ← loaded: /kaggle/input/datasets/himanshubendale000/himanshuuu/m1_3arm_fits.csv

Above chance: 23/23


In [43]:
# ===== Cell 8 — M1 parameter recovery (synthetic ground-truth check) =====
from scipy.stats import pearsonr

def simulate_3arm(alpha, beta, n_trials, reward_probs, seed):
    rng = np.random.default_rng(seed)
    Q = np.array([0.5, 0.5, 0.5])
    choices, rewards = [], []
    for t in range(n_trials):
        exp_q = np.exp(np.clip(beta * (Q - Q.max()), -500, 500))
        probs = exp_q / exp_q.sum()
        ch = rng.choice(3, p=probs)
        rw = 1 if rng.random() < reward_probs[t, ch] else 0
        Q[ch] += alpha * (rw - Q[ch])
        choices.append(ch); rewards.append(rw)
    return pd.DataFrame({"chosen_stim": np.array(choices) + 1, "reward": rewards})

def make_oscillating_probs(n_trials, meanintercept=0.55, amplitude=0.35):
    t = np.arange(1, n_trials + 1)
    def osc(phase): return meanintercept + amplitude * np.real(np.exp(2j * np.pi * 0.025 * t + phase))
    return np.column_stack([np.clip(osc(0), .05, .95),
                             np.clip(osc(2*np.pi/3), .05, .95),
                             np.clip(osc(4*np.pi/3), .05, .95)])

def _build_m1_recovery(n_synth=30, n_trials=479, seed_base=999):
    rng = np.random.default_rng(seed_base)
    rows = []
    for i in range(n_synth):
        true_alpha, true_beta = rng.uniform(0.05, 0.95), rng.uniform(0.5, 10)
        rp = make_oscillating_probs(n_trials)
        synth = simulate_3arm(true_alpha, true_beta, n_trials, rp, seed=seed_base + i)
        rec_x, _ = fit_model_3arm("m1", synth, n_starts=10, seed=seed_base + i)
        rows.append({"synth_id": i, "true_alpha": true_alpha, "true_beta": true_beta,
                     "rec_alpha": rec_x[0], "rec_beta": rec_x[1]})
        if (i + 1) % 10 == 0:
            print(f"  recovery {i+1}/{n_synth} done")
    return pd.DataFrame(rows)

recovery_m1 = cached("m1_3arm_recovery.csv", _build_m1_recovery)
for k in ["alpha", "beta"]:
    r, p = pearsonr(recovery_m1[f"true_{k}"], recovery_m1[f"rec_{k}"])
    print(f"  {k}: r={r:.3f} (p={p:.4f})")

✔ m1_3arm_recovery.csv                     ← loaded: /kaggle/input/datasets/himanshubendale000/himanshuuu/m1_3arm_recovery.csv
  alpha: r=0.982 (p=0.0000)
  beta: r=0.985 (p=0.0000)


In [44]:
# ===== Cell 9 — Fit M2 and M3 on all subjects (per-subject checkpointed) =====
def _build_fits(model):
    def _fit_one(s):
        t = all_trials[s]
        x, nll = fit_model_3arm(model, t)
        chance_nll = len(t) * np.log(3)
        row = {"subject": s, "n_trials": len(t), "nll": nll,
               "chance_nll": chance_nll, "above_chance": nll < chance_nll - 5}
        row.update(dict(zip(PARAMS[model], x)))
        print(f"  {s}: nll={nll:.2f}, above_chance={row['above_chance']}")
        return row

    rows_dict = cached_incremental(f"{model}_3arm_fits_by_subject.pkl", all_subjects, _fit_one)
    return pd.DataFrame([rows_dict[s] for s in all_subjects])

m2_fits_3arm = _build_fits("m2")
m3_fits_3arm = _build_fits("m3")

print(f"\nM2 above-chance: {m2_fits_3arm['above_chance'].sum()}/23")
print(f"M3 above-chance: {m3_fits_3arm['above_chance'].sum()}/23")

  sub-001: nll=346.09, above_chance=True
  sub-002: nll=174.37, above_chance=True
  sub-003: nll=152.16, above_chance=True
  sub-004: nll=265.46, above_chance=True
  sub-005: nll=173.35, above_chance=True
  sub-006: nll=201.74, above_chance=True
  sub-007: nll=200.85, above_chance=True
  sub-008: nll=196.10, above_chance=True
  sub-009: nll=303.05, above_chance=True
  sub-010: nll=128.39, above_chance=True
  sub-011: nll=127.64, above_chance=True
  sub-012: nll=248.40, above_chance=True
  sub-013: nll=135.22, above_chance=True
  sub-014: nll=162.05, above_chance=True
  sub-015: nll=346.57, above_chance=True
  sub-016: nll=313.10, above_chance=True
  sub-017: nll=205.68, above_chance=True
  sub-018: nll=118.51, above_chance=True
  sub-019: nll=173.46, above_chance=True
  sub-020: nll=280.04, above_chance=True
  sub-021: nll=319.77, above_chance=True
  sub-022: nll=145.78, above_chance=True
  sub-023: nll=368.60, above_chance=True
💾 m2_3arm_fits_by_subject.pkl: 23/23 keys complete → /kag

In [45]:
# ===== Cell 10 — AIC model comparison: M1 vs M2 vs M3 =====
from scipy.stats import ttest_1samp

def _build_aic_comparison():
    rows = []
    for s in all_subjects:
        nll1 = fits_df.loc[fits_df["subject"] == s, "nll"].iloc[0]
        nll2 = m2_fits_3arm.loc[m2_fits_3arm["subject"] == s, "nll"].iloc[0]
        nll3 = m3_fits_3arm.loc[m3_fits_3arm["subject"] == s, "nll"].iloc[0]
        aic1, aic2, aic3 = 2*2 + 2*nll1, 2*3 + 2*nll2, 2*4 + 2*nll3
        winner = min([("M1", aic1), ("M2", aic2), ("M3", aic3)], key=lambda x: x[1])[0]
        rows.append({"subject": s, "aic_m1": aic1, "aic_m2": aic2, "aic_m3": aic3, "winner": winner})
    return pd.DataFrame(rows)

aic_df = cached("aic_3arm_comparison.csv", _build_aic_comparison)
print(aic_df.round(2).to_string(index=False))
print("\nWinner counts:\n", aic_df["winner"].value_counts().to_string())

print("\nPairwise mean|ΔAIC| (mimicry threshold: <4 = mimicry present)")
for a, b in [("aic_m1", "aic_m2"), ("aic_m1", "aic_m3"), ("aic_m2", "aic_m3")]:
    d = aic_df[a] - aic_df[b]
    t, p = ttest_1samp(d, 0)
    verdict = "mimicry present" if d.abs().mean() < 4 else "mimicry absent"
    print(f"  {a} vs {b}: mean|Δ|={d.abs().mean():.3f}, t={t:.3f}, p={p:.4f} → {verdict}")

✔ aic_3arm_comparison.csv                  ← loaded: /kaggle/working/aic_3arm_comparison.csv
subject  aic_m1  aic_m2  aic_m3 winner
sub-001  697.29  698.18  701.01     M1
sub-002  352.75  354.74  352.20     M3
sub-003  309.97  310.33  313.97     M1
sub-004  534.92  536.92  538.92     M1
sub-005  350.69  352.69  354.55     M1
sub-006  444.65  409.48  448.65     M2
sub-007  406.00  407.69  389.31     M3
sub-008  399.53  398.20  398.50     M2
sub-009  610.81  612.09  611.48     M1
sub-010  260.77  262.77  264.64     M1
sub-011  261.18  261.28  264.40     M1
sub-012  527.04  502.81  531.04     M2
sub-013  275.02  276.44  260.40     M3
sub-014  349.73  330.10  338.90     M2
sub-015  698.19  699.15  701.57     M1
sub-016  635.00  632.20  628.93     M3
sub-017  415.36  417.36  418.70     M1
sub-018  251.12  243.02  246.83     M2
sub-019  351.65  352.92  353.24     M1
sub-020  577.51  566.07  581.51     M2
sub-021  651.40  645.55  647.53     M2
sub-022  296.08  297.56  298.64     M1
sub-023  7

# ===== B9 — M1 (3-arm) parameter recovery — synthetic data (safety step before real EEG) =====
# doc 3 Section 7 discipline: first prove the fitting pipeline itself is reliable,
# only then connect those fits to EEG.

from scipy.stats import pearsonr

def simulate_3arm(alpha, beta, n_trials, reward_probs_over_time, seed):
    """reward_probs_over_time: (n_trials, 3) array -- the true win-probability of each arm at each trial"""
    rng = np.random.default_rng(seed)
    Q = np.array([0.5, 0.5, 0.5])
    choices, rewards = [], []
    for t in range(n_trials):
        exp_q = np.exp(np.clip(beta * (Q - Q.max()), -500, 500))
        probs = exp_q / exp_q.sum()
        ch = rng.choice(3, p=probs)
        rw = 1 if rng.random() < reward_probs_over_time[t, ch] else 0
        rpe = rw - Q[ch]
        Q[ch] += alpha * rpe
        choices.append(ch); rewards.append(rw)
    return pd.DataFrame({"chosen_stim": np.array(choices) + 1, "reward": rewards})

def make_oscillating_probs(n_trials, meanintercept=0.55, amplitude=0.35):
    """Replicate the oscillating-probability logic from the dataset's README/BEH_BANDIT.m"""
    t = np.arange(1, n_trials + 1)
    def osc(phase_shift):
        return meanintercept + amplitude * np.real(np.exp(2j * np.pi * 0.025 * t + phase_shift))
    p1 = np.clip(osc(0), 0.05, 0.95)
    p2 = np.clip(osc(2 * np.pi / 3), 0.05, 0.95)
    p3 = np.clip(osc(4 * np.pi / 3), 0.05, 0.95)
    return np.column_stack([p1, p2, p3])

def build_m1_recovery(n_synth=30, n_trials=479, seed_base=999):
    rng = np.random.default_rng(seed_base)
    rows = []
    for i in range(n_synth):
        true_alpha = rng.uniform(0.05, 0.95)
        true_beta = rng.uniform(0.5, 10)
        rp = make_oscillating_probs(n_trials)
        synth = simulate_3arm(true_alpha, true_beta, n_trials, rp, seed=seed_base + i)
        rec_x, _ = fit_m1_3arm(synth, n_starts=10, seed=seed_base + i)
        rows.append({"synth_id": i, "true_alpha": true_alpha, "true_beta": true_beta,
                     "rec_alpha": rec_x[0], "rec_beta": rec_x[1]})
        if (i + 1) % 10 == 0:
            print(f"  recovery {i+1}/{n_synth} done")
    return pd.DataFrame(rows)

print("M1 (3-arm) parameter recovery running (~30 synthetic subjects)...")
recovery_m1 = build_m1_recovery()

print("\n=== RECOVERY RESULTS ===")
for k in ["alpha", "beta"]:
    r, p = pearsonr(recovery_m1[f"true_{k}"], recovery_m1[f"rec_{k}"])
    verdict = "✅ good" if r > 0.7 else ("⚠️ weak" if r > 0.3 else "❌ poor")
    print(f"  {k}: r={r:.3f}  (p={p:.4f})  {verdict}")

# ---- boundary-clustering check: do synthetic subjects with high alpha (>0.9) recover correctly ----
high_true = recovery_m1[recovery_m1["true_alpha"] > 0.8]
print(f"\n=== high true_alpha (>0.8) subjects (n={len(high_true)}) -- do they recover correctly? ===")
print(high_true[["true_alpha", "rec_alpha"]].round(3).to_string(index=False))

recovery_m1.to_csv("m1_3arm_recovery.csv", index=False)
print("\nSaved: m1_3arm_recovery.csv")

In [46]:
# ===== Cell 11 — Out-of-sample check (odd/even trial split) =====
def fit_predict_oos(model, trials_df, seed=42):
    n = len(trials_df)
    idx = np.arange(n)
    train_idx, test_idx = idx[idx % 2 == 0], idx[idx % 2 == 1]
    train_df = trials_df.iloc[train_idx].reset_index(drop=True)
    test_df = trials_df.iloc[test_idx].reset_index(drop=True)

    x, _ = fit_model_3arm(model, train_df, n_starts=10, seed=seed)
    ch_test = (test_df["chosen_stim"].values - 1).astype(int)
    rw_test = test_df["reward"].values.astype(float)
    nll_test, _ = run_model_3arm(model, x, ch_test, rw_test)
    return nll_test

def _build_oos_comparison():
    rows = []
    for s in all_subjects:
        t = all_trials[s]
        row = {"subject": s}
        for model in ("m1", "m2", "m3"):
            row[f"oos_nll_{model}"] = fit_predict_oos(model, t)
        rows.append(row)
        print(f"  {s}: M1={row['oos_nll_m1']:.2f}, M2={row['oos_nll_m2']:.2f}, M3={row['oos_nll_m3']:.2f}")
    return pd.DataFrame(rows)

oos_df = cached("oos_3arm_comparison.csv", _build_oos_comparison)
oos_df["winner"] = oos_df[["oos_nll_m1", "oos_nll_m2", "oos_nll_m3"]].idxmin(axis=1)
print("\nOOS winner counts:\n", oos_df["winner"].value_counts().to_string())
for a, b in [("m1", "m2"), ("m1", "m3"), ("m2", "m3")]:
    d = oos_df[f"oos_nll_{a}"] - oos_df[f"oos_nll_{b}"]
    t, p = ttest_1samp(d, 0)
    print(f"  {a.upper()} vs {b.upper()}: mean_diff={d.mean():+.3f}, t={t:.3f}, p={p:.4f}")

✔ oos_3arm_comparison.csv                  ← loaded: /kaggle/working/oos_3arm_comparison.csv

OOS winner counts:
 winner
oos_nll_m2    10
oos_nll_m1     7
oos_nll_m3     6
  M1 vs M2: mean_diff=-0.206, t=-0.221, p=0.8274
  M1 vs M3: mean_diff=-0.112, t=-0.227, p=0.8224
  M2 vs M3: mean_diff=+0.095, t=0.084, p=0.9338


In [47]:
# ===== Cell 12 — Feedback-locked epoching: high-pass filter + baseline correction =====
# v2 pipeline (final, correct version): raw EEG was unfiltered in the first attempt,
# which let slow drift dominate the signal (per-trial std ~1e-5, no usable structure).
# Fix: 0.5 Hz high-pass filter on the continuous raw data, then per-trial baseline
# subtraction using the [-0.2, 0.0]s pre-feedback window. Trigger-code logic below
# (STIM +100 offset, dynamic find_event_code) was verified against the events.tsv
# parse on sub-001: 100% match on both chosen_stim and reward.

import mne
mne.set_log_level("ERROR")

EPOCH_TMIN, EPOCH_TMAX = -0.2, 0.8
BASELINE_TMIN, BASELINE_TMAX = -0.2, 0.0

def epoch_one_subject_v2(subj):
    set_key = f"{DS}/{subj}/eeg/{subj}_task-ThreeArmedBandit_eeg.set"
    fdt_key = f"{DS}/{subj}/eeg/{subj}_task-ThreeArmedBandit_eeg.fdt"
    set_path, fdt_path = OUT / f"{subj}_eeg.set", OUT / f"{subj}_eeg.fdt"

    s3.download_file(BUCKET, set_key, str(set_path))
    s3.download_file(BUCKET, fdt_key, str(fdt_path))

    raw_s = mne.io.read_raw_eeglab(str(set_path), preload=True, verbose=False)
    raw_s.filter(l_freq=0.5, h_freq=None, verbose=False)

    events_s, event_id_s = mne.events_from_annotations(raw_s, verbose=False)
    fb_win = find_event_code(event_id_s, 5)
    fb_loss = find_event_code(event_id_s, 6)
    resp_codes = {find_event_code(event_id_s, n): p for n, p in [(1, "Up"), (2, "Left"), (3, "Right")]}
    stim_codes = {find_event_code(event_id_s, n): n for n in STIM_TRIGGER_NUMS}

    rows = []
    pending_stim_map = pending_choice = stim_sample = resp_sample = None
    for samp, _, code_ in events_s:
        if code_ in stim_codes:
            d3 = str(stim_codes[code_] + 100)
            up, left, right = int(d3[0]), int(d3[1]), int(d3[2])
            pending_stim_map = {"Up": up, "Left": left, "Right": right}
            pending_choice, stim_sample, resp_sample = None, samp, None
        elif code_ in resp_codes:
            resp_sample = samp
            if pending_stim_map is not None:
                pending_choice = pending_stim_map.get(resp_codes[code_])
        elif code_ in (fb_win, fb_loss):
            reward = 1 if code_ == fb_win else 0
            if pending_choice is not None:
                rows.append({"feedback_sample": samp, "chosen_stim": pending_choice, "reward": reward})
            pending_stim_map = pending_choice = stim_sample = resp_sample = None
    trial_meta_s = pd.DataFrame(rows)

    sfreq = raw_s.info["sfreq"]
    picks = {reg: mne.pick_channels(raw_s.ch_names, chs)
             for reg, chs in [("frontal", FRONTAL), ("central", CENTRAL), ("parieto_occipital", PARIETO_OCCIPITAL)]}
    data = raw_s.get_data()
    n_pre, n_post = int(abs(EPOCH_TMIN) * sfreq), int(EPOCH_TMAX * sfreq)
    n_base_pre = int(abs(BASELINE_TMIN) * sfreq)

    region_epochs = {reg: [] for reg in REGIONS}
    valid_idx = []
    for i, samp in enumerate(trial_meta_s["feedback_sample"].values):
        start, end = samp - n_pre, samp + n_post
        base_start = samp - n_base_pre
        if start < 0 or end > data.shape[1] or base_start < 0:
            continue
        for reg in REGIONS:
            trial_wave = data[picks[reg], start:end].mean(axis=0)
            baseline_val = data[picks[reg], base_start:samp].mean()
            region_epochs[reg].append(trial_wave - baseline_val)
        valid_idx.append(i)

    trial_meta_s = trial_meta_s.iloc[valid_idx].reset_index(drop=True)
    for reg in REGIONS:
        region_epochs[reg] = np.stack(region_epochs[reg])

    del raw_s, data
    gc.collect()
    set_path.unlink(missing_ok=True)
    fdt_path.unlink(missing_ok=True)

    return {"trial_meta": trial_meta_s, "epochs": region_epochs, "sfreq": sfreq}


all_epochs_v2 = cached_incremental("all_subjects_epochs_ds003458_v2.pkl", all_subjects, epoch_one_subject_v2)

for s, d in all_epochs_v2.items():
    print(f"  {s}: {len(d['trial_meta'])} trials, frontal shape={d['epochs']['frontal'].shape}, "
          f"amp std={d['epochs']['frontal'].std():.2e}")

✔ all_subjects_epochs_ds003458_v2.pkl      ← resuming from: /kaggle/working/all_subjects_epochs_ds003458_v2.pkl (23/23 keys already done)
💾 all_subjects_epochs_ds003458_v2.pkl: 23/23 keys complete → /kaggle/working/all_subjects_epochs_ds003458_v2.pkl
  sub-001: 479 trials, frontal shape=(479, 500), amp std=3.72e-05
  sub-002: 478 trials, frontal shape=(478, 500), amp std=3.99e-05
  sub-003: 480 trials, frontal shape=(480, 500), amp std=9.39e-05
  sub-004: 478 trials, frontal shape=(478, 500), amp std=3.58e-05
  sub-005: 480 trials, frontal shape=(480, 500), amp std=2.68e-05
  sub-006: 480 trials, frontal shape=(480, 500), amp std=1.97e-05
  sub-007: 479 trials, frontal shape=(479, 500), amp std=2.94e-05
  sub-008: 478 trials, frontal shape=(478, 500), amp std=4.24e-05
  sub-009: 479 trials, frontal shape=(479, 500), amp std=2.34e-05
  sub-010: 477 trials, frontal shape=(477, 500), amp std=2.30e-05
  sub-011: 480 trials, frontal shape=(480, 500), amp std=2.04e-05
  sub-012: 480 trials, 

In [48]:
# ===== Cell 13 — Per-trial RPE traces (M1, M3) from fitted params, forward-pass =====
def compute_rpe_trace(model, params, trials_df):
    choices = (trials_df["chosen_stim"].values - 1).astype(int)
    rewards = trials_df["reward"].values.astype(float)
    n = len(choices)
    Q = np.array([0.5, 0.5, 0.5]); A = np.array([0.5, 0.5, 0.5])
    rpes = np.zeros(n)
    for t in range(n):
        ch = choices[t]
        rpe = rewards[t] - Q[ch]
        rpes[t] = rpe
        if model == "m1":
            Q[ch] += params[0] * rpe
        else:  # m3
            alpha0, kappa, eta = params
            a_t = np.clip(alpha0 + kappa * A[ch], 0.001, 0.999)
            Q[ch] += a_t * rpe
            A[ch] = eta * abs(rpe) + (1 - eta) * A[ch]
    return rpes

def _build_one_rpe_trace(s):
    t = all_trials[s]
    m1_row = fits_df[fits_df["subject"] == s].iloc[0]
    rpe_m1 = compute_rpe_trace("m1", [m1_row["alpha"]], t)
    m3_row = m3_fits_3arm[m3_fits_3arm["subject"] == s].iloc[0]
    rpe_m3 = compute_rpe_trace("m3", [m3_row["alpha0"], m3_row["kappa"], m3_row["eta"]], t)
    return pd.DataFrame({"trial_num": t["trial_num"].values, "chosen_stim": t["chosen_stim"].values,
                          "reward": t["reward"].values, "rpe_m1": rpe_m1, "rpe_m3": rpe_m3})

rpe_traces = cached_incremental("rpe_traces_ds003458.pkl", all_subjects, _build_one_rpe_trace)
print(rpe_traces["sub-001"].head(10).to_string())

✔ rpe_traces_ds003458.pkl                  ← resuming from: /kaggle/working/rpe_traces_ds003458.pkl (23/23 keys already done)
💾 rpe_traces_ds003458.pkl: 23/23 keys complete → /kaggle/working/rpe_traces_ds003458.pkl
   trial_num  chosen_stim  reward    rpe_m1        rpe_m3
0          1            3       1  0.500000  5.000000e-01
1          2            2       1  0.500000  5.000000e-01
2          3            3       0 -0.994852 -9.995000e-01
3          4            2       0 -0.994852 -9.995000e-01
4          5            2       0 -0.010242 -9.995000e-04
5          6            1       1  0.500000  5.000000e-01
6          7            1       1  0.005148  5.000000e-04
7          8            2       0 -0.000105 -9.995000e-07
8          9            3       1  0.989758  9.990005e-01
9         10            2       0 -0.000001 -9.995000e-10


In [49]:
# ===== Cell 14 — Merge RPE traces with v2 epochs (with strict alignment check) =====
def merge_subject_v2(s):
    rpe_df = rpe_traces[s].reset_index(drop=True)
    epoch_data = all_epochs_v2[s]
    meta_df = epoch_data["trial_meta"].reset_index(drop=True)

    if len(rpe_df) != len(meta_df):
        return {"status": f"MISMATCH: rpe={len(rpe_df)}, epoch={len(meta_df)}"}

    match_stim = (rpe_df["chosen_stim"].values == meta_df["chosen_stim"].values).mean()
    match_rew = (rpe_df["reward"].values == meta_df["reward"].values).mean()
    if match_stim < 1.0 or match_rew < 1.0:
        return {"status": f"ALIGNMENT MISMATCH: stim={match_stim:.2%}, reward={match_rew:.2%}"}

    return {"status": "OK", "trial_meta": rpe_df.copy(),
            "epochs": epoch_data["epochs"], "sfreq": epoch_data["sfreq"]}

_merged_raw = cached_incremental("merged_dataset_ds003458_v2_raw.pkl", all_subjects, merge_subject_v2)

merged_dataset_v2 = {s: d for s, d in _merged_raw.items() if d["status"] == "OK"}
issues = {s: d["status"] for s, d in _merged_raw.items() if d["status"] != "OK"}
if issues:
    print(f"⚠ {len(issues)} subjects had issues: {issues}")
print(f"\n✔ merged: {len(merged_dataset_v2)}/{len(all_subjects)} subjects ready for CNN decoding")

💾 merged_dataset_ds003458_v2_raw.pkl: 23/23 keys complete → /kaggle/working/merged_dataset_ds003458_v2_raw.pkl

✔ merged: 23/23 subjects ready for CNN decoding


In [50]:
# ===== Cell 15 — CNN decoder + out-of-sample RPE decoding (per subject × region × model) =====
import torch, torch.nn as nn
from sklearn.model_selection import KFold

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RPE_MODELS = ["m1", "m3"]  # M2 excluded — consistently null in ds004295

class RPE_CNN(nn.Module):
    def __init__(self, n_samples):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, 25, padding=12), nn.ReLU(), nn.BatchNorm1d(16), nn.MaxPool1d(4),
            nn.Conv1d(16, 32, 15, padding=7), nn.ReLU(), nn.BatchNorm1d(32), nn.MaxPool1d(4),
            nn.Conv1d(32, 32, 7, padding=3), nn.ReLU(), nn.BatchNorm1d(32), nn.AdaptiveAvgPool1d(1),
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        return self.fc(self.conv(x).squeeze(-1)).squeeze(-1)


def train_cnn_fold(X_train, y_train, X_val, n_epochs=60, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    model = RPE_CNN(X_train.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    Xt = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1).to(DEVICE)
    yt = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)
    Xv = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1).to(DEVICE)
    mu, sigma = Xt.mean(), Xt.std() + 1e-8
    Xt, Xv = (Xt - mu) / sigma, (Xv - mu) / sigma

    model.train()
    for _ in range(n_epochs):
        opt.zero_grad()
        loss = loss_fn(model(Xt), yt)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        return model(Xv).cpu().numpy()


def decode_subject_region_model(subj_data, region, rpe_col, n_folds=5, seed=0):
    X = subj_data["epochs"][region]
    y = subj_data["trial_meta"][rpe_col].values
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    pred_rpe = np.full(len(y), np.nan)
    for train_idx, val_idx in kf.split(X):
        pred_rpe[val_idx] = train_cnn_fold(X[train_idx], y[train_idx], X[val_idx], seed=seed)
    return pred_rpe


def _build_one_decode(key, seed=0):
    region, model_name = key.split("__")
    rpe_col = f"rpe_{model_name}"
    preds, rs = {}, []
    for s, data in merged_dataset_v2.items():
        pred_rpe = decode_subject_region_model(data, region, rpe_col, seed=seed)
        r = np.corrcoef(pred_rpe, data["trial_meta"][rpe_col].values)[0, 1]
        preds[s] = pred_rpe
        rs.append(r)
        print(f"    {s}: decode r={r:.3f}")
    print(f"  {key}: mean_r={np.mean(rs):.3f}, sd={np.std(rs):.3f}")
    return preds

decode_keys = [f"{region}__{model}" for region in REGIONS for model in RPE_MODELS]
cnn_predictions_v2 = cached_incremental(
    "cnn_predictions_ds003458_v2_seed0.pkl", decode_keys,
    lambda k: _build_one_decode(k, seed=0),
)

print("\n=== Summary: mean decode-r by region×model ===")
for key, preds in cnn_predictions_v2.items():
    region, model_name = key.split("__")
    rpe_col = f"rpe_{model_name}"
    rs = [np.corrcoef(preds[s], merged_dataset_v2[s]["trial_meta"][rpe_col].values)[0, 1]
          for s in preds]
    print(f"  {key}: mean_r={np.mean(rs):.3f}, sd={np.std(rs):.3f}")

    sub-001: decode r=-0.011
    sub-002: decode r=0.102
    sub-003: decode r=-0.057
    sub-004: decode r=0.100
    sub-005: decode r=0.100
    sub-006: decode r=0.265
    sub-007: decode r=0.098
    sub-008: decode r=0.062
    sub-009: decode r=0.132
    sub-010: decode r=0.158
    sub-011: decode r=0.063
    sub-012: decode r=0.207
    sub-013: decode r=0.223
    sub-014: decode r=0.156
    sub-015: decode r=0.116
    sub-016: decode r=0.141
    sub-017: decode r=0.071
    sub-018: decode r=0.297
    sub-019: decode r=0.071
    sub-020: decode r=0.042
    sub-021: decode r=0.006
    sub-022: decode r=0.223
    sub-023: decode r=0.156
  frontal__m1: mean_r=0.118, sd=0.085
    sub-001: decode r=-0.033
    sub-002: decode r=0.044
    sub-003: decode r=-0.053
    sub-004: decode r=0.091
    sub-005: decode r=0.150
    sub-006: decode r=0.203
    sub-007: decode r=0.138
    sub-008: decode r=0.072
    sub-009: decode r=0.122
    sub-010: decode r=0.127
    sub-011: decode r=0.045
    su

In [54]:
# ===== Cell 16 — Multi-seed decoding, parieto-occipital only (feeds into interaction test) =====
# Re-uses train_cnn_fold / decode_subject_region_model from Cell 15.
# Only PO region needed here — frontal/central already showed the pattern in ds004295
# to be secondary; PO is where B6's interaction test lives.

SEEDS = [0, 1, 2, 3]
PO_MODELS = ["m1", "m3"]

def _build_one_multiseed(key):
    region, model_name, seed = key.split("__")
    seed = int(seed)
    rpe_col = f"rpe_{model_name}"
    preds = {}
    for s, data in merged_dataset_v2.items():
        preds[s] = decode_subject_region_model(data, region, rpe_col, seed=seed)
    return preds

multiseed_keys = [f"parieto_occipital__{m}__{sd}" for m in PO_MODELS for sd in SEEDS]
cnn_predictions_po_multiseed = cached_incremental(
    "cnn_predictions_ds003458_po_multiseed.pkl", multiseed_keys, _build_one_multiseed
)

print("✔ multi-seed PO decoding done:", list(cnn_predictions_po_multiseed.keys()))

💾 cnn_predictions_ds003458_po_multiseed.pkl: 8/8 keys complete → /kaggle/working/cnn_predictions_ds003458_po_multiseed.pkl
✔ multi-seed PO decoding done: ['parieto_occipital__m1__0', 'parieto_occipital__m1__1', 'parieto_occipital__m1__2', 'parieto_occipital__m1__3', 'parieto_occipital__m3__0', 'parieto_occipital__m3__1', 'parieto_occipital__m3__2', 'parieto_occipital__m3__3']


In [55]:
# ===== Cell 17 — predicted_rpe × reward interaction test (PO, M1 & M3) =====
# Logic (mirrors ds004295 B6):
#   1. Per subject, per seed: regress decoded_rpe ~ true_rpe + reward + true_rpe:reward
#      -> interaction beta captures whether the CNN's decoded signal carries the
#         reward-conditioned (signed) RPE structure, not just raw valence.
#   2. Average interaction beta across seeds -> one number per subject per model.
#   3. One-sample t-test across 23 subjects (is mean beta != 0).
#   4. Subject-level jackknife: drop-one-subject, recompute t/p, check the effect
#      isn't driven by 1-2 subjects.
#   5. FDR correction across the model-level tests (M1, M3).

import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

def _subject_interaction_beta(s, model_name, seed):
    key = f"parieto_occipital__{model_name}__{seed}"
    decoded = cnn_predictions_po_multiseed[key][s]
    data = merged_dataset_v2[s]
    true_rpe = data["trial_meta"][f"rpe_{model_name}"].values
    reward = data["trial_meta"]["reward"].values.astype(float)

    X = np.column_stack([true_rpe, reward, true_rpe * reward])
    X = sm.add_constant(X)
    valid = ~np.isnan(decoded)
    model = sm.OLS(decoded[valid], X[valid]).fit()
    return model.params[3]  # interaction term coefficient

def _build_interaction_betas(model_name):
    rows = []
    for s in all_subjects:
        betas = [_subject_interaction_beta(s, model_name, sd) for sd in SEEDS]
        rows.append({"subject": s, "mean_beta": np.mean(betas), "sd_beta": np.std(betas)})
    return pd.DataFrame(rows)

interaction_results = {}
for model_name in PO_MODELS:
    interaction_results[model_name] = cached(
        f"po_interaction_betas_{model_name}.csv",
        lambda mn=model_name: _build_interaction_betas(mn)
    )

print("=== Interaction β summary (per model) ===")
main_test_rows = []
for model_name, df in interaction_results.items():
    t, p = ttest_1samp(df["mean_beta"], 0)
    main_test_rows.append({"model": model_name, "mean_beta": df["mean_beta"].mean(),
                            "t": t, "p": p})
    print(f"  {model_name.upper()}: mean_beta={df['mean_beta'].mean():.4f}, t={t:.3f}, p={p:.4f}")

main_test_df = pd.DataFrame(main_test_rows)
rej, p_fdr, _, _ = multipletests(main_test_df["p"], alpha=0.05, method="fdr_bh")
main_test_df["p_fdr"] = p_fdr
main_test_df["significant_fdr"] = rej
print("\n=== FDR-corrected ===")
print(main_test_df.round(4).to_string(index=False))

print("\n=== Subject-jackknife robustness (leave-one-out t-test) ===")
for model_name, df in interaction_results.items():
    betas = df["mean_beta"].values
    loo_ps = []
    for i in range(len(betas)):
        loo = np.delete(betas, i)
        _, p_loo = ttest_1samp(loo, 0)
        loo_ps.append(p_loo)
    n_robust = sum(p < 0.05 for p in loo_ps)
    print(f"  {model_name.upper()}: {n_robust}/{len(betas)} leave-one-out runs stay p<0.05 "
          f"(max p across drops = {max(loo_ps):.4f})")

… building po_interaction_betas_m1.csv (not found in cache)
💾 saved po_interaction_betas_m1.csv → /kaggle/working/po_interaction_betas_m1.csv
… building po_interaction_betas_m3.csv (not found in cache)
💾 saved po_interaction_betas_m3.csv → /kaggle/working/po_interaction_betas_m3.csv
=== Interaction β summary (per model) ===
  M1: mean_beta=-0.0168, t=-1.407, p=0.1735
  M3: mean_beta=-0.0166, t=-1.403, p=0.1745

=== FDR-corrected ===
model  mean_beta       t      p  p_fdr  significant_fdr
   m1    -0.0168 -1.4067 0.1735 0.1745            False
   m3    -0.0166 -1.4033 0.1745 0.1745            False

=== Subject-jackknife robustness (leave-one-out t-test) ===
  M1: 0/23 leave-one-out runs stay p<0.05 (max p across drops = 0.3141)
  M3: 0/23 leave-one-out runs stay p<0.05 (max p across drops = 0.3203)
